# Layers of defence for an agent

A real kagent incident-response agent calls Claude through agentgateway and uses
operations tools deployed by kagent. It can investigate incidents, close them all,
or delete their investigation history. Tool calls really change a disposable dataset.
Each chapter adds a control and checks what changed.
Agent turns use A2A and show the returned tool-call trace. Direct HTTP and MCP requests
are labelled as boundary checks and run from the kagent agent's managed runtime.
There is no stand-in agent pod, echo model or hand-deployed tool server.

This lab uses Enterprise kagent, Enterprise agentgateway and Istio on mesh1.
Model calls use the installed Anthropic credential and are billable. Prompts use
specimen data, responses are capped and request loops are bounded.
Kindnet does not enforce NetworkPolicy, so the egress chapter controls example.com
through a namespace-scoped Istio waypoint. It does not claim a blanket internet boundary.

## Connect

Set the environment when following the notebook directly. The console sets it for every step.

In [ ]:
export DD=demo-scripts/defence DD_URL=http://dd-gateway.dd-gateway.svc
export DD_STATE="${TMPDIR:-/tmp}/defence-lab"
. "$DD/helpers.sh"

## 1. Create an incident-response agent in kagent

Build an incident-response agent in kagent and connect it to three open incidents. First, show that it can close every incident without fixing the faults. Then add policies that let it investigate but prevent those changes.

kagent creates and manages the runtime. The model is Claude, reached through agentgateway. The provider credential stays at the gateway; the agent gets its own lab identity.

<svg font-family="Arial, sans-serif" id="defence-architecture" width="100%" xmlns="http://www.w3.org/2000/svg" class="flowchart" style="max-width: 649.03515625px;" viewBox="0 0 649.03515625 705.334228515625" role="graphics-document document" aria-roledescription="flowchart-v2"><style>#defence-architecture{font-size:20px;fill:#1e293b;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#defence-architecture .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#defence-architecture .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#defence-architecture .error-icon{fill:#ffffff;}#defence-architecture .error-text{fill:#000000;stroke:#000000;}#defence-architecture .edge-thickness-normal{stroke-width:1px;}#defence-architecture .edge-thickness-thick{stroke-width:3.5px;}#defence-architecture .edge-pattern-solid{stroke-dasharray:0;}#defence-architecture .edge-thickness-invisible{stroke-width:0;fill:none;}#defence-architecture .edge-pattern-dashed{stroke-dasharray:3;}#defence-architecture .edge-pattern-dotted{stroke-dasharray:2;}#defence-architecture .marker{fill:#64748b;stroke:#64748b;}#defence-architecture .marker.cross{stroke:#64748b;}#defence-architecture svg{font-size:20px;}#defence-architecture p{margin:0;}#defence-architecture .label{color:#1e293b;}#defence-architecture .cluster-label text{fill:#000000;}#defence-architecture .cluster-label span{color:#000000;}#defence-architecture .cluster-label span p{background-color:transparent;}#defence-architecture .label text,#defence-architecture span{fill:#1e293b;color:#1e293b;}#defence-architecture .node rect,#defence-architecture .node circle,#defence-architecture .node ellipse,#defence-architecture .node polygon,#defence-architecture .node path{fill:#f5f3ff;stroke:#8b5cf6;stroke-width:1px;}#defence-architecture .rough-node .label text,#defence-architecture .node .label text,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-anchor:middle;}#defence-architecture .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#defence-architecture .rough-node .label,#defence-architecture .node .label,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-align:center;}#defence-architecture .node.clickable{cursor:pointer;}#defence-architecture .root .anchor path{fill:#64748b!important;stroke-width:0;stroke:#64748b;}#defence-architecture .arrowheadPath{fill:#0b0b0b;}#defence-architecture .edgePath .path{stroke:#64748b;stroke-width:2.0px;}#defence-architecture .flowchart-link{stroke:#64748b;fill:none;}#defence-architecture .edgeLabel{background-color:#eff6ff;text-align:center;}#defence-architecture .edgeLabel p{background-color:#eff6ff;}#defence-architecture .edgeLabel rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .labelBkg{background-color:rgba(239, 246, 255, 0.5);}#defence-architecture .cluster rect{fill:#f8fafc;stroke:#cbd5e1;stroke-width:1px;}#defence-architecture .cluster text{fill:#000000;}#defence-architecture .cluster span{color:#000000;}#defence-architecture div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-size:12px;background:#ffffff;border:1px solid hsl(0, 0%, 90%);border-radius:2px;pointer-events:none;z-index:100;}#defence-architecture .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#1e293b;}#defence-architecture rect.text{fill:none;stroke-width:0;}#defence-architecture .icon-shape,#defence-architecture .image-shape{background-color:#eff6ff;text-align:center;}#defence-architecture .icon-shape p,#defence-architecture .image-shape p{background-color:#eff6ff;padding:2px;}#defence-architecture .icon-shape rect,#defence-architecture .image-shape rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#defence-architecture .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#defence-architecture :root{--mermaid-font-family:Helvetica,"Liberation Sans",Arial,"Hiragino Kaku Gothic ProN","Noto Sans CJK JP","Microsoft YaHei","Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif;}#defence-architecture .solo rect{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo polygon{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo ellipse{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo circle{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo path{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo tspan{fill:#312e81!important;}#defence-architecture .data rect{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data polygon{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data ellipse{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data circle{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data path{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data tspan{fill:#7c2d12!important;}#defence-architecture .external rect{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external polygon{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external ellipse{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external circle{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external path{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external tspan{fill:#1e3a8a!important;}</style><g><marker id="defence-architecture_flowchart-v2-pointEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-pointStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="4.5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-circleEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="11" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-circleStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="-1" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="12" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="-1" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M310.665,223L248.09,255.5L248.09,284" id="L_agent_gateway_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_gateway_0" data-points="W3sieCI6MzEwLjY2NTAzOTA2MjUsInkiOjIyM30seyJ4IjoyNDguMDg5ODQzNzUsInkiOjI1NS41fSx7IngiOjI0OC4wODk4NDM3NSwieSI6Mjg4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M172.882,385L122.484,417.5L122.484,446" id="L_gateway_tools_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_tools_0" data-points="W3sieCI6MTcyLjg4MTYzMDk3OTkzODI4LCJ5IjozODV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDE3LjV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDUwfV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M122.484,525L122.484,557.5L122.484,586" id="L_tools_records_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_tools_records_0" data-points="W3sieCI6MTIyLjQ4NDM3NSwieSI6NTI1fSx7IngiOjEyMi40ODQzNzUsInkiOjU1Ny41fSx7IngiOjEyMi40ODQzNzUsInkiOjU5MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M455.069,223L517.645,255.5L517.645,295" id="L_agent_egress_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_egress_0" data-points="W3sieCI6NDU1LjA2OTMzNTkzNzUsInkiOjIyM30seyJ4Ijo1MTcuNjQ0NTMxMjUsInkiOjI1NS41fSx7IngiOjUxNy42NDQ1MzEyNSwieSI6Mjk5fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M382.867,83L382.867,115.5L382.867,144" id="L_user_agent_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_user_agent_0" data-points="W3sieCI6MzgyLjg2NzE4NzUsInkiOjgzfSx7IngiOjM4Mi44NjcxODc1LCJ5IjoxMTUuNX0seyJ4IjozODIuODY3MTg3NSwieSI6MTQ4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M323.298,385L373.695,417.5L373.695,446" id="L_gateway_claude_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_claude_0" data-points="W3sieCI6MzIzLjI5ODA1NjUyMDA2MTcsInkiOjM4NX0seyJ4IjozNzMuNjk1MzEyNSwieSI6NDE3LjV9LHsieCI6MzczLjY5NTMxMjUsInkiOjQ1MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(248.08984375, 255.5)"><g class="label" data-id="L_agent_gateway_0" transform="translate(-82.4140625, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="164.828125" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCP</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 417.5)"><g class="label" data-id="L_gateway_tools_0" transform="translate(-92.0546875, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="184.109375" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> workload</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 557.5)"><g class="label" data-id="L_tools_records_0" transform="translate(-90.9609375, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="181.921875" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Reads</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> changes</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(517.64453125, 255.5)"><g class="label" data-id="L_agent_egress_0" transform="translate(-99.828125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="199.65625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Outside-access</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(382.8671875, 115.5)"><g class="label" data-id="L_user_agent_0" transform="translate(-75.953125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="151.90625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">A2A</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> via</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> gateway</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(373.6953125, 417.5)"><g class="label" data-id="L_gateway_claude_0" transform="translate(-52.03125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="104.0625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g></g><g class="nodes"><g class="node default external" id="flowchart-user-0" transform="translate(382.8671875, 45.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-120.6171875" y="-37.5" width="241.234375" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Operator</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Investigate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incidents</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-agent-1" transform="translate(382.8671875, 185.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-108.953125" y="-37.5" width="217.90625" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Agent</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> response</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-gateway-2" transform="translate(248.08984375, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-116.1640625" y="-48.5" width="232.328125" height="97"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -33.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">agentgateway</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">JWT</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tool</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> access</tspan></tspan><tspan class="text-outer-tspan" x="0" y="2.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Content</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> rate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> limits</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-tools-3" transform="translate(122.484375, 487.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-114.484375" y="-37.5" width="228.96875" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCPServer</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tools</tspan></tspan></text></g></g></g><g class="node default data" id="flowchart-records-4" transform="translate(122.484375, 643.6671180725098)"><path d="M0,15.781710914454276 a107,15.781710914454276 0,0,0 214,0 a107,15.781710914454276 0,0,0 -214,0 l0,75.78171091445428 a107,15.781710914454276 0,0,0 214,0 l0,-75.78171091445428" class="basic label-container" style="fill:#fff7ed !important;stroke:#fb923c !important" transform="translate(-107, -53.67256637168141)"></path><g class="label" style="color:#7c2d12 !important" transform="translate(0, -14.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#7c2d12 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Three</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> records</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Status</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> history</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-egress-5" transform="translate(517.64453125, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-123.390625" y="-37.5" width="246.78125" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> egress</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> waypoint</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Deny</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> example.com</tspan></tspan></text></g></g></g><g class="node default external" id="flowchart-claude-17" transform="translate(373.6953125, 487.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-106.7265625" y="-37.5" width="213.453125" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Claude</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Haiku</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> 4.5</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Anthropic</tspan></tspan></text></g></g></g></g></g></g></svg>

### Create the namespaces

Enrol the lab's agent, tool and gateway namespaces in ambient.

**Expected:** Three dd- namespaces are created.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: v1
kind: Namespace
metadata:
  name: dd-agents
  labels: {istio.io/dataplane-mode: ambient}
---
apiVersion: v1
kind: Namespace
metadata:
  name: dd-tools
  labels: {istio.io/dataplane-mode: ambient}
---
apiVersion: v1
kind: Namespace
metadata:
  name: dd-gateway
  labels: {istio.io/dataplane-mode: ambient}
YAML

### Give the agent an identity

Create the lab signing key and store Alice's gateway token in a Secret for kagent to reference. Copy the installed Anthropic credential into the lab gateway's namespace without printing it.

**Expected:** The agent identity and gateway provider credential are stored separately.

In [ ]:
python3 "$DD/identity.py" setup
ALICE=$(python3 "$DD/identity.py" token --user alice)
kubectl --context kind-mesh1 -n dd-agents create secret generic dd-agent-identity --from-literal="token=$ALICE" --from-literal="authorization=Bearer $ALICE" --dry-run=client -o yaml | kubectl --context kind-mesh1 apply -f -
kubectl --context kind-mesh1 -n agentgateway-system get secret anthropic-secret -o json | jq '{apiVersion:"v1",kind:"Secret",metadata:{name:"dd-anthropic",namespace:"dd-gateway"},type:"Opaque",data:.data}' | kubectl --context kind-mesh1 apply -f -

### Run the incident tool server

Build the server for reading and changing the lab's incident records, then let kagent deploy it.

**Expected:** The image's record-mutation tests pass and defence-operations reports Ready.

In [ ]:
docker build -t localhost:5001/defence-ops-mcp:v1 "$DD/operations-mcp"
docker push localhost:5001/defence-ops-mcp:v1
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: kagent.dev/v1alpha1
kind: MCPServer
metadata: {name: defence-operations, namespace: dd-tools}
spec:
  deployment:
    image: localhost:5001/defence-ops-mcp:v1
    imagePullPolicy: Always
    port: 3000
    resources:
      requests: {cpu: 10m, memory: 32Mi}
      limits: {cpu: 200m, memory: 128Mi}
  transportType: http
  httpTransport: {path: /mcp, targetPort: 3000}
YAML
kubectl --context kind-mesh1 -n dd-tools wait mcpserver/defence-operations --for=condition=Ready --timeout=120s

### Route model, tool and agent traffic

Create the lab gateway. The model backend calls Anthropic, the MCP backend reaches the kagent-managed server, and the A2A route exposes the agent's runtime.

**Expected:** dd-gateway rolls out. The agent route starts serving after the next step creates the Agent.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: {name: dd-gateway, namespace: dd-gateway}
spec:
  gatewayClassName: enterprise-agentgateway
  listeners:
  - name: http
    protocol: HTTP
    port: 80
    allowedRoutes: {namespaces: {from: Same}}
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: {name: model, namespace: dd-gateway}
spec:
  ai:
    provider:
      anthropic: {model: claude-haiku-4-5}
  policies:
    auth:
      secretRef: {name: dd-anthropic}
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: {name: tools, namespace: dd-gateway}
spec:
  mcp:
    targets:
    - name: operations
      static: {host: defence-operations.dd-tools.svc.cluster.local, port: 3000, path: /mcp}
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: model, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: PathPrefix, value: /v1/chat/completions}}]
    backendRefs:
    - {name: model, group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend}
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: tools, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: PathPrefix, value: /mcp}}]
    backendRefs:
    - {name: tools, group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend}
---
# The lab client reaches the real kagent agent through its A2A endpoint.
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: agent, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: PathPrefix, value: /a2a}}]
    filters:
    - type: URLRewrite
      urlRewrite:
        path: {type: ReplacePrefixMatch, replacePrefixMatch: /}
    backendRefs:
    - {name: defence-agent, namespace: dd-agents, port: 8080}
---
apiVersion: gateway.networking.k8s.io/v1beta1
kind: ReferenceGrant
metadata: {name: dd-gateway-to-agent, namespace: dd-agents}
spec:
  from:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, namespace: dd-gateway}
  to:
  - {group: "", kind: Service, name: defence-agent}
---
# Read-only incident state for the console. JWT authentication on the gateway
# applies here too. Mutations are MCP tools, not public HTTP write endpoints.
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: incident-state, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: Exact, value: /ops-state}}]
    filters:
    - type: URLRewrite
      urlRewrite:
        path: {type: ReplaceFullPath, replaceFullPath: /state}
    backendRefs:
    - {name: defence-operations, namespace: dd-tools, port: 3000}
---
apiVersion: gateway.networking.k8s.io/v1beta1
kind: ReferenceGrant
metadata: {name: dd-gateway-to-state, namespace: dd-tools}
spec:
  from:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, namespace: dd-gateway}
  to:
  - {group: "", kind: Service, name: defence-operations}
YAML
kubectl --context kind-mesh1 -n dd-gateway rollout status deploy/dd-gateway --timeout=120s

### Create the kagent Agent

The ModelConfig uses agentgateway's OpenAI-compatible endpoint. The RemoteMCPServer registers the gateway's tools. The Agent references both.

**Expected:** kagent accepts the Agent and its runtime becomes Ready.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: kagent.dev/v1alpha2
kind: ModelConfig
metadata: {name: defence-model, namespace: dd-agents}
spec:
  provider: OpenAI
  model: claude-haiku-4-5
  apiKeySecret: dd-agent-identity
  apiKeySecretKey: token
  openAI:
    baseUrl: http://dd-gateway.dd-gateway.svc/v1
    maxTokens: 256
    temperature: "0"
---
apiVersion: kagent.dev/v1alpha2
kind: RemoteMCPServer
metadata: {name: defence-tools, namespace: dd-agents}
spec:
  description: Incident operations reached through agentgateway
  protocol: STREAMABLE_HTTP
  url: http://dd-gateway.dd-gateway.svc/mcp
  headersFrom:
  - name: Authorization
    valueFrom: {type: Secret, name: dd-agent-identity, key: authorization}
---
apiVersion: kagent.dev/v1alpha2
kind: Agent
metadata: {name: defence-agent, namespace: dd-agents}
spec:
  type: Declarative
  description: An incident-response agent whose operational permissions are enforced by agentgateway
  declarative:
    modelConfig: defence-model
    runtime: python
    systemMessage: |
      You are an incident-response assistant. Use tools to investigate incidents
      and carry out requested operations on the incident records.
      These are disposable lab records. Execute explicitly requested operations
      when the tool is available, without asking for another confirmation.
      For incident questions, always read the records with a tool. For an action,
      call the named tool and report what it actually changed. Do not claim that
      closing an incident resolves its underlying fault. Do not invent records.
      If a requested tool is unavailable or a call is refused, say so and stop.
      Do not substitute another tool or claim the operation succeeded.
      For a request to repeat text or reply with a word, reply directly.
      Keep replies to one paragraph.
    tools:
    - type: McpServer
      mcpServer:
        kind: RemoteMCPServer
        apiGroup: kagent.dev
        name: defence-tools
        toolNames: [list_incidents, get_incident, close_all_incidents, delete_incident_history]
    deployment:
      resources:
        requests: {cpu: 25m, memory: 128Mi}
        limits: {cpu: 500m, memory: 512Mi}
YAML
kubectl --context kind-mesh1 -n dd-agents wait agent/defence-agent --for=condition=Ready --timeout=180s

### Ask the agent to investigate

Send an A2A message asking for the current incidents. Read the actual tool call and records returned by the agent, then check that kagent owns its Deployment.

**Expected:** The agent reads three open incidents affecting payments, customer-api and settlement. Its Deployment is owned by Agent/defence-agent.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
kubectl --context kind-mesh1 -n dd-agents get deployment defence-agent -o json | jq '{owner:.metadata.ownerReferences[0].kind,name:.metadata.ownerReferences[0].name,serviceAccount:.spec.template.spec.serviceAccountName}'

### Declare the outside destination

Put example.com behind an Istio waypoint visible only to dd-agents. This is a direct egress check from the kagent runtime, not an agent turn.

**Expected:** The waypoint is Ready and HTTPS to example.com returns 200.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
# Only this namespace sees the ServiceEntry. Other labs keep their DNS answers.
# This covers example.com, not arbitrary internet destinations or IP literals.
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: dd-egress
  namespace: dd-agents
  labels: {istio.io/waypoint-for: service}
spec:
  gatewayClassName: istio-waypoint
  listeners: [{name: mesh, port: 15008, protocol: HBONE}]
---
apiVersion: networking.istio.io/v1
kind: ServiceEntry
metadata:
  name: outside-example
  namespace: dd-agents
  labels: {istio.io/use-waypoint: dd-egress}
spec:
  exportTo: ["."]
  hosts: [example.com]
  ports: [{number: 443, name: tls, protocol: TLS}]
  resolution: DNS
  location: MESH_EXTERNAL
YAML
kubectl --context kind-mesh1 -n dd-agents rollout status deploy/dd-egress --timeout=120s
kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 --retry 10 --retry-all-errors --retry-delay 1 -o /dev/null -w 'Outside HTTP %{http_code}\n' https://example.com

### Select the installed rate limiter

The default Service on this rig also selects the waypoint limiter, which has a separate counter. Create a lab-owned Service selecting the main limiter's active ReplicaSet.

**Expected:** dd-rate-limiter selects the main limiter without changing its Deployment.

In [ ]:
HASH=$(kubectl --context kind-mesh1 -n agentgateway-system get rs -o json | jq -er '[.items[] | select(any(.metadata.ownerReferences[]?; .kind == "Deployment" and .name == "rate-limiter-enterprise-agentgateway")) | select(.spec.replicas > 0)] | sort_by(.metadata.creationTimestamp) | last | .metadata.labels["pod-template-hash"]')
jq -n --arg hash "$HASH" '{apiVersion:"v1",kind:"Service",metadata:{name:"dd-rate-limiter",namespace:"agentgateway-system"},spec:{selector:{app:"rate-limiter","pod-template-hash":$hash},ports:[{name:"grpc",port:8083,targetPort:8083}]}}' | kubectl --context kind-mesh1 apply -f -
kubectl --context kind-mesh1 -n agentgateway-system get service dd-rate-limiter -o yaml

## 2. Control the agent's outside access

Check whether the kagent runtime can reach example.com. Add a deny policy to its egress waypoint, then repeat the request.

This checks a workload boundary directly. The request is not generated by the model. Kindnet does not enforce NetworkPolicy, so this chapter proves control of example.com only.

<svg font-family="Arial, sans-serif" id="defence-architecture" width="100%" xmlns="http://www.w3.org/2000/svg" class="flowchart" style="max-width: 649.03515625px;" viewBox="0 0 649.03515625 705.334228515625" role="graphics-document document" aria-roledescription="flowchart-v2"><style>#defence-architecture{font-size:20px;fill:#1e293b;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#defence-architecture .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#defence-architecture .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#defence-architecture .error-icon{fill:#ffffff;}#defence-architecture .error-text{fill:#000000;stroke:#000000;}#defence-architecture .edge-thickness-normal{stroke-width:1px;}#defence-architecture .edge-thickness-thick{stroke-width:3.5px;}#defence-architecture .edge-pattern-solid{stroke-dasharray:0;}#defence-architecture .edge-thickness-invisible{stroke-width:0;fill:none;}#defence-architecture .edge-pattern-dashed{stroke-dasharray:3;}#defence-architecture .edge-pattern-dotted{stroke-dasharray:2;}#defence-architecture .marker{fill:#64748b;stroke:#64748b;}#defence-architecture .marker.cross{stroke:#64748b;}#defence-architecture svg{font-size:20px;}#defence-architecture p{margin:0;}#defence-architecture .label{color:#1e293b;}#defence-architecture .cluster-label text{fill:#000000;}#defence-architecture .cluster-label span{color:#000000;}#defence-architecture .cluster-label span p{background-color:transparent;}#defence-architecture .label text,#defence-architecture span{fill:#1e293b;color:#1e293b;}#defence-architecture .node rect,#defence-architecture .node circle,#defence-architecture .node ellipse,#defence-architecture .node polygon,#defence-architecture .node path{fill:#f5f3ff;stroke:#8b5cf6;stroke-width:1px;}#defence-architecture .rough-node .label text,#defence-architecture .node .label text,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-anchor:middle;}#defence-architecture .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#defence-architecture .rough-node .label,#defence-architecture .node .label,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-align:center;}#defence-architecture .node.clickable{cursor:pointer;}#defence-architecture .root .anchor path{fill:#64748b!important;stroke-width:0;stroke:#64748b;}#defence-architecture .arrowheadPath{fill:#0b0b0b;}#defence-architecture .edgePath .path{stroke:#64748b;stroke-width:2.0px;}#defence-architecture .flowchart-link{stroke:#64748b;fill:none;}#defence-architecture .edgeLabel{background-color:#eff6ff;text-align:center;}#defence-architecture .edgeLabel p{background-color:#eff6ff;}#defence-architecture .edgeLabel rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .labelBkg{background-color:rgba(239, 246, 255, 0.5);}#defence-architecture .cluster rect{fill:#f8fafc;stroke:#cbd5e1;stroke-width:1px;}#defence-architecture .cluster text{fill:#000000;}#defence-architecture .cluster span{color:#000000;}#defence-architecture div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-size:12px;background:#ffffff;border:1px solid hsl(0, 0%, 90%);border-radius:2px;pointer-events:none;z-index:100;}#defence-architecture .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#1e293b;}#defence-architecture rect.text{fill:none;stroke-width:0;}#defence-architecture .icon-shape,#defence-architecture .image-shape{background-color:#eff6ff;text-align:center;}#defence-architecture .icon-shape p,#defence-architecture .image-shape p{background-color:#eff6ff;padding:2px;}#defence-architecture .icon-shape rect,#defence-architecture .image-shape rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#defence-architecture .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#defence-architecture :root{--mermaid-font-family:Helvetica,"Liberation Sans",Arial,"Hiragino Kaku Gothic ProN","Noto Sans CJK JP","Microsoft YaHei","Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif;}#defence-architecture .solo rect{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo polygon{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo ellipse{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo circle{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo path{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo tspan{fill:#312e81!important;}#defence-architecture .data rect{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data polygon{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data ellipse{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data circle{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data path{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data tspan{fill:#7c2d12!important;}#defence-architecture .external rect{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external polygon{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external ellipse{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external circle{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external path{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external tspan{fill:#1e3a8a!important;}</style><g><marker id="defence-architecture_flowchart-v2-pointEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-pointStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="4.5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-circleEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="11" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-circleStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="-1" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="12" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="-1" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M310.665,223L248.09,255.5L248.09,284" id="L_agent_gateway_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_gateway_0" data-points="W3sieCI6MzEwLjY2NTAzOTA2MjUsInkiOjIyM30seyJ4IjoyNDguMDg5ODQzNzUsInkiOjI1NS41fSx7IngiOjI0OC4wODk4NDM3NSwieSI6Mjg4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M172.882,385L122.484,417.5L122.484,446" id="L_gateway_tools_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_tools_0" data-points="W3sieCI6MTcyLjg4MTYzMDk3OTkzODI4LCJ5IjozODV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDE3LjV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDUwfV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M122.484,525L122.484,557.5L122.484,586" id="L_tools_records_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_tools_records_0" data-points="W3sieCI6MTIyLjQ4NDM3NSwieSI6NTI1fSx7IngiOjEyMi40ODQzNzUsInkiOjU1Ny41fSx7IngiOjEyMi40ODQzNzUsInkiOjU5MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M455.069,223L517.645,255.5L517.645,295" id="L_agent_egress_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_egress_0" data-points="W3sieCI6NDU1LjA2OTMzNTkzNzUsInkiOjIyM30seyJ4Ijo1MTcuNjQ0NTMxMjUsInkiOjI1NS41fSx7IngiOjUxNy42NDQ1MzEyNSwieSI6Mjk5fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M382.867,83L382.867,115.5L382.867,144" id="L_user_agent_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_user_agent_0" data-points="W3sieCI6MzgyLjg2NzE4NzUsInkiOjgzfSx7IngiOjM4Mi44NjcxODc1LCJ5IjoxMTUuNX0seyJ4IjozODIuODY3MTg3NSwieSI6MTQ4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M323.298,385L373.695,417.5L373.695,446" id="L_gateway_claude_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_claude_0" data-points="W3sieCI6MzIzLjI5ODA1NjUyMDA2MTcsInkiOjM4NX0seyJ4IjozNzMuNjk1MzEyNSwieSI6NDE3LjV9LHsieCI6MzczLjY5NTMxMjUsInkiOjQ1MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(248.08984375, 255.5)"><g class="label" data-id="L_agent_gateway_0" transform="translate(-82.4140625, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="164.828125" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCP</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 417.5)"><g class="label" data-id="L_gateway_tools_0" transform="translate(-92.0546875, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="184.109375" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> workload</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 557.5)"><g class="label" data-id="L_tools_records_0" transform="translate(-90.9609375, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="181.921875" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Reads</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> changes</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(517.64453125, 255.5)"><g class="label" data-id="L_agent_egress_0" transform="translate(-99.828125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="199.65625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Outside-access</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(382.8671875, 115.5)"><g class="label" data-id="L_user_agent_0" transform="translate(-75.953125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="151.90625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">A2A</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> via</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> gateway</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(373.6953125, 417.5)"><g class="label" data-id="L_gateway_claude_0" transform="translate(-52.03125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="104.0625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g></g><g class="nodes"><g class="node default external" id="flowchart-user-0" transform="translate(382.8671875, 45.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-120.6171875" y="-37.5" width="241.234375" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Operator</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Investigate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incidents</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-agent-1" transform="translate(382.8671875, 185.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-108.953125" y="-37.5" width="217.90625" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Agent</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> response</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-gateway-2" transform="translate(248.08984375, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-116.1640625" y="-48.5" width="232.328125" height="97"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -33.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">agentgateway</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">JWT</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tool</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> access</tspan></tspan><tspan class="text-outer-tspan" x="0" y="2.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Content</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> rate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> limits</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-tools-3" transform="translate(122.484375, 487.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-114.484375" y="-37.5" width="228.96875" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCPServer</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tools</tspan></tspan></text></g></g></g><g class="node default data" id="flowchart-records-4" transform="translate(122.484375, 643.6671180725098)"><path d="M0,15.781710914454276 a107,15.781710914454276 0,0,0 214,0 a107,15.781710914454276 0,0,0 -214,0 l0,75.78171091445428 a107,15.781710914454276 0,0,0 214,0 l0,-75.78171091445428" class="basic label-container" style="fill:#fff7ed !important;stroke:#fb923c !important" transform="translate(-107, -53.67256637168141)"></path><g class="label" style="color:#7c2d12 !important" transform="translate(0, -14.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#7c2d12 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Three</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> records</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Status</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> history</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-egress-5" transform="translate(517.64453125, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-123.390625" y="-37.5" width="246.78125" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> egress</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> waypoint</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Deny</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> example.com</tspan></tspan></text></g></g></g><g class="node default external" id="flowchart-claude-17" transform="translate(373.6953125, 487.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-106.7265625" y="-37.5" width="213.453125" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Claude</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Haiku</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> 4.5</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Anthropic</tspan></tspan></text></g></g></g></g></g></g></svg>

### Without the control

Fetch example.com from the kagent agent's managed runtime.

**Expected:** The HTTPS request returns 200.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 -o /dev/null -w 'Outside HTTP %{http_code}\n' https://example.com

### Add the control

Deny traffic handled by this lab's egress waypoint.

**Expected:** The AuthorizationPolicy is applied.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: {name: outside-deny, namespace: dd-agents}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: Gateway, name: dd-egress}
  action: DENY
  rules: [{}]
YAML
sleep 2

### With the control

Repeat the direct egress check and read the waypoint's RBAC counter. Then ask the agent to investigate incidents through its permitted model and tool paths.

**Expected:** Outside access is refused and recorded. The agent still completes its tool call.

In [ ]:
if kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 -o /dev/null https://example.com; then printf 'Unexpected outside access\n'; exit 1; else printf 'Outside request refused\n'; fi
kubectl --context kind-mesh1 -n dd-agents exec deploy/dd-egress -- pilot-agent request GET 'stats?filter=rbac'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

## 3. Stop the agent bypassing the tool gateway

The incident tools are managed by kagent. Without a workload policy, the agent runtime can also call them directly. Require agent traffic to pass through agentgateway.

Istio checks SPIFFE identity. The gateway is admitted, and kagent's controller keeps direct access for tool discovery. The agent's own identity is refused on the direct path.

<svg font-family="Arial, sans-serif" id="defence-architecture" width="100%" xmlns="http://www.w3.org/2000/svg" class="flowchart" style="max-width: 649.03515625px;" viewBox="0 0 649.03515625 705.334228515625" role="graphics-document document" aria-roledescription="flowchart-v2"><style>#defence-architecture{font-size:20px;fill:#1e293b;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#defence-architecture .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#defence-architecture .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#defence-architecture .error-icon{fill:#ffffff;}#defence-architecture .error-text{fill:#000000;stroke:#000000;}#defence-architecture .edge-thickness-normal{stroke-width:1px;}#defence-architecture .edge-thickness-thick{stroke-width:3.5px;}#defence-architecture .edge-pattern-solid{stroke-dasharray:0;}#defence-architecture .edge-thickness-invisible{stroke-width:0;fill:none;}#defence-architecture .edge-pattern-dashed{stroke-dasharray:3;}#defence-architecture .edge-pattern-dotted{stroke-dasharray:2;}#defence-architecture .marker{fill:#64748b;stroke:#64748b;}#defence-architecture .marker.cross{stroke:#64748b;}#defence-architecture svg{font-size:20px;}#defence-architecture p{margin:0;}#defence-architecture .label{color:#1e293b;}#defence-architecture .cluster-label text{fill:#000000;}#defence-architecture .cluster-label span{color:#000000;}#defence-architecture .cluster-label span p{background-color:transparent;}#defence-architecture .label text,#defence-architecture span{fill:#1e293b;color:#1e293b;}#defence-architecture .node rect,#defence-architecture .node circle,#defence-architecture .node ellipse,#defence-architecture .node polygon,#defence-architecture .node path{fill:#f5f3ff;stroke:#8b5cf6;stroke-width:1px;}#defence-architecture .rough-node .label text,#defence-architecture .node .label text,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-anchor:middle;}#defence-architecture .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#defence-architecture .rough-node .label,#defence-architecture .node .label,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-align:center;}#defence-architecture .node.clickable{cursor:pointer;}#defence-architecture .root .anchor path{fill:#64748b!important;stroke-width:0;stroke:#64748b;}#defence-architecture .arrowheadPath{fill:#0b0b0b;}#defence-architecture .edgePath .path{stroke:#64748b;stroke-width:2.0px;}#defence-architecture .flowchart-link{stroke:#64748b;fill:none;}#defence-architecture .edgeLabel{background-color:#eff6ff;text-align:center;}#defence-architecture .edgeLabel p{background-color:#eff6ff;}#defence-architecture .edgeLabel rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .labelBkg{background-color:rgba(239, 246, 255, 0.5);}#defence-architecture .cluster rect{fill:#f8fafc;stroke:#cbd5e1;stroke-width:1px;}#defence-architecture .cluster text{fill:#000000;}#defence-architecture .cluster span{color:#000000;}#defence-architecture div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-size:12px;background:#ffffff;border:1px solid hsl(0, 0%, 90%);border-radius:2px;pointer-events:none;z-index:100;}#defence-architecture .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#1e293b;}#defence-architecture rect.text{fill:none;stroke-width:0;}#defence-architecture .icon-shape,#defence-architecture .image-shape{background-color:#eff6ff;text-align:center;}#defence-architecture .icon-shape p,#defence-architecture .image-shape p{background-color:#eff6ff;padding:2px;}#defence-architecture .icon-shape rect,#defence-architecture .image-shape rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#defence-architecture .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#defence-architecture :root{--mermaid-font-family:Helvetica,"Liberation Sans",Arial,"Hiragino Kaku Gothic ProN","Noto Sans CJK JP","Microsoft YaHei","Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif;}#defence-architecture .solo rect{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo polygon{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo ellipse{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo circle{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo path{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo tspan{fill:#312e81!important;}#defence-architecture .data rect{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data polygon{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data ellipse{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data circle{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data path{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data tspan{fill:#7c2d12!important;}#defence-architecture .external rect{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external polygon{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external ellipse{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external circle{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external path{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external tspan{fill:#1e3a8a!important;}</style><g><marker id="defence-architecture_flowchart-v2-pointEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-pointStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="4.5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-circleEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="11" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-circleStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="-1" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="12" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="-1" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M310.665,223L248.09,255.5L248.09,284" id="L_agent_gateway_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_gateway_0" data-points="W3sieCI6MzEwLjY2NTAzOTA2MjUsInkiOjIyM30seyJ4IjoyNDguMDg5ODQzNzUsInkiOjI1NS41fSx7IngiOjI0OC4wODk4NDM3NSwieSI6Mjg4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M172.882,385L122.484,417.5L122.484,446" id="L_gateway_tools_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_tools_0" data-points="W3sieCI6MTcyLjg4MTYzMDk3OTkzODI4LCJ5IjozODV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDE3LjV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDUwfV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M122.484,525L122.484,557.5L122.484,586" id="L_tools_records_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_tools_records_0" data-points="W3sieCI6MTIyLjQ4NDM3NSwieSI6NTI1fSx7IngiOjEyMi40ODQzNzUsInkiOjU1Ny41fSx7IngiOjEyMi40ODQzNzUsInkiOjU5MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M455.069,223L517.645,255.5L517.645,295" id="L_agent_egress_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_egress_0" data-points="W3sieCI6NDU1LjA2OTMzNTkzNzUsInkiOjIyM30seyJ4Ijo1MTcuNjQ0NTMxMjUsInkiOjI1NS41fSx7IngiOjUxNy42NDQ1MzEyNSwieSI6Mjk5fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M382.867,83L382.867,115.5L382.867,144" id="L_user_agent_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_user_agent_0" data-points="W3sieCI6MzgyLjg2NzE4NzUsInkiOjgzfSx7IngiOjM4Mi44NjcxODc1LCJ5IjoxMTUuNX0seyJ4IjozODIuODY3MTg3NSwieSI6MTQ4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M323.298,385L373.695,417.5L373.695,446" id="L_gateway_claude_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_claude_0" data-points="W3sieCI6MzIzLjI5ODA1NjUyMDA2MTcsInkiOjM4NX0seyJ4IjozNzMuNjk1MzEyNSwieSI6NDE3LjV9LHsieCI6MzczLjY5NTMxMjUsInkiOjQ1MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(248.08984375, 255.5)"><g class="label" data-id="L_agent_gateway_0" transform="translate(-82.4140625, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="164.828125" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCP</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 417.5)"><g class="label" data-id="L_gateway_tools_0" transform="translate(-92.0546875, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="184.109375" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> workload</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 557.5)"><g class="label" data-id="L_tools_records_0" transform="translate(-90.9609375, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="181.921875" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Reads</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> changes</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(517.64453125, 255.5)"><g class="label" data-id="L_agent_egress_0" transform="translate(-99.828125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="199.65625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Outside-access</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(382.8671875, 115.5)"><g class="label" data-id="L_user_agent_0" transform="translate(-75.953125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="151.90625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">A2A</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> via</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> gateway</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(373.6953125, 417.5)"><g class="label" data-id="L_gateway_claude_0" transform="translate(-52.03125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="104.0625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g></g><g class="nodes"><g class="node default external" id="flowchart-user-0" transform="translate(382.8671875, 45.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-120.6171875" y="-37.5" width="241.234375" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Operator</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Investigate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incidents</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-agent-1" transform="translate(382.8671875, 185.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-108.953125" y="-37.5" width="217.90625" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Agent</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> response</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-gateway-2" transform="translate(248.08984375, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-116.1640625" y="-48.5" width="232.328125" height="97"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -33.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">agentgateway</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">JWT</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tool</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> access</tspan></tspan><tspan class="text-outer-tspan" x="0" y="2.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Content</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> rate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> limits</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-tools-3" transform="translate(122.484375, 487.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-114.484375" y="-37.5" width="228.96875" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCPServer</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tools</tspan></tspan></text></g></g></g><g class="node default data" id="flowchart-records-4" transform="translate(122.484375, 643.6671180725098)"><path d="M0,15.781710914454276 a107,15.781710914454276 0,0,0 214,0 a107,15.781710914454276 0,0,0 -214,0 l0,75.78171091445428 a107,15.781710914454276 0,0,0 214,0 l0,-75.78171091445428" class="basic label-container" style="fill:#fff7ed !important;stroke:#fb923c !important" transform="translate(-107, -53.67256637168141)"></path><g class="label" style="color:#7c2d12 !important" transform="translate(0, -14.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#7c2d12 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Three</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> records</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Status</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> history</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-egress-5" transform="translate(517.64453125, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-123.390625" y="-37.5" width="246.78125" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> egress</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> waypoint</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Deny</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> example.com</tspan></tspan></text></g></g></g><g class="node default external" id="flowchart-claude-17" transform="translate(373.6953125, 487.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-106.7265625" y="-37.5" width="213.453125" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Claude</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Haiku</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> 4.5</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Anthropic</tspan></tspan></text></g></g></g></g></g></g></svg>

### Without the control

Make a direct MCP call from the kagent runtime to the operations Service, bypassing agentgateway.

**Expected:** The incident records are returned directly.

In [ ]:
dd_mcp "$ALICE" tools/call '{"name":"list_incidents","arguments":{}}' http://defence-operations.dd-tools.svc:3000/mcp

### Add the control

Allow the gateway and kagent's discovery controller into the tool namespace. Do not admit the agent's ServiceAccount.

**Expected:** gateway-only is applied in dd-tools.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
# Agent traffic must pass through agentgateway. kagent's controller also needs
# direct access to discover the tools of the MCPServer it manages.
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: {name: gateway-only, namespace: dd-tools}
spec:
  action: ALLOW
  rules:
  - from:
    - source:
        principals:
        - mesh1/ns/dd-gateway/sa/dd-gateway
        - mesh1/ns/kagent/sa/kagent-controller
YAML
sleep 2

### With the control

Repeat the direct MCP call and read the ztunnel refusal. Then ask the agent to investigate through its configured gateway path.

**Expected:** Direct access is refused under the defence-agent SPIFFE identity. The agent's normal tool call still reads the incidents.

In [ ]:
if dd_mcp "$ALICE" tools/call '{"name":"list_incidents","arguments":{}}' http://defence-operations.dd-tools.svc:3000/mcp; then printf 'Unexpected direct tool access\n'; exit 1; else printf 'Direct tool request refused\n'; fi
kubectl --context kind-mesh1 -n istio-system logs -l app=ztunnel --since=30s --tail=3000 | jq -c 'select(.["src.namespace"] == "dd-agents" and .direction == "inbound" and (.error // "" | contains("policy rejection"))) | {source:.["src.identity"],destination:.["dst.namespace"],error}'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

## 4. Require a signed caller token

The gateway currently accepts a model request without a token. Add Strict JWT authentication, then compare the anonymous request with the agent's authenticated turn.

The agent's ModelConfig and RemoteMCPServer reference its identity Secret. The Anthropic provider key is held only by agentgateway.

<svg font-family="Arial, sans-serif" id="defence-architecture" width="100%" xmlns="http://www.w3.org/2000/svg" class="flowchart" style="max-width: 649.03515625px;" viewBox="0 0 649.03515625 705.334228515625" role="graphics-document document" aria-roledescription="flowchart-v2"><style>#defence-architecture{font-size:20px;fill:#1e293b;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#defence-architecture .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#defence-architecture .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#defence-architecture .error-icon{fill:#ffffff;}#defence-architecture .error-text{fill:#000000;stroke:#000000;}#defence-architecture .edge-thickness-normal{stroke-width:1px;}#defence-architecture .edge-thickness-thick{stroke-width:3.5px;}#defence-architecture .edge-pattern-solid{stroke-dasharray:0;}#defence-architecture .edge-thickness-invisible{stroke-width:0;fill:none;}#defence-architecture .edge-pattern-dashed{stroke-dasharray:3;}#defence-architecture .edge-pattern-dotted{stroke-dasharray:2;}#defence-architecture .marker{fill:#64748b;stroke:#64748b;}#defence-architecture .marker.cross{stroke:#64748b;}#defence-architecture svg{font-size:20px;}#defence-architecture p{margin:0;}#defence-architecture .label{color:#1e293b;}#defence-architecture .cluster-label text{fill:#000000;}#defence-architecture .cluster-label span{color:#000000;}#defence-architecture .cluster-label span p{background-color:transparent;}#defence-architecture .label text,#defence-architecture span{fill:#1e293b;color:#1e293b;}#defence-architecture .node rect,#defence-architecture .node circle,#defence-architecture .node ellipse,#defence-architecture .node polygon,#defence-architecture .node path{fill:#f5f3ff;stroke:#8b5cf6;stroke-width:1px;}#defence-architecture .rough-node .label text,#defence-architecture .node .label text,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-anchor:middle;}#defence-architecture .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#defence-architecture .rough-node .label,#defence-architecture .node .label,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-align:center;}#defence-architecture .node.clickable{cursor:pointer;}#defence-architecture .root .anchor path{fill:#64748b!important;stroke-width:0;stroke:#64748b;}#defence-architecture .arrowheadPath{fill:#0b0b0b;}#defence-architecture .edgePath .path{stroke:#64748b;stroke-width:2.0px;}#defence-architecture .flowchart-link{stroke:#64748b;fill:none;}#defence-architecture .edgeLabel{background-color:#eff6ff;text-align:center;}#defence-architecture .edgeLabel p{background-color:#eff6ff;}#defence-architecture .edgeLabel rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .labelBkg{background-color:rgba(239, 246, 255, 0.5);}#defence-architecture .cluster rect{fill:#f8fafc;stroke:#cbd5e1;stroke-width:1px;}#defence-architecture .cluster text{fill:#000000;}#defence-architecture .cluster span{color:#000000;}#defence-architecture div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-size:12px;background:#ffffff;border:1px solid hsl(0, 0%, 90%);border-radius:2px;pointer-events:none;z-index:100;}#defence-architecture .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#1e293b;}#defence-architecture rect.text{fill:none;stroke-width:0;}#defence-architecture .icon-shape,#defence-architecture .image-shape{background-color:#eff6ff;text-align:center;}#defence-architecture .icon-shape p,#defence-architecture .image-shape p{background-color:#eff6ff;padding:2px;}#defence-architecture .icon-shape rect,#defence-architecture .image-shape rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#defence-architecture .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#defence-architecture :root{--mermaid-font-family:Helvetica,"Liberation Sans",Arial,"Hiragino Kaku Gothic ProN","Noto Sans CJK JP","Microsoft YaHei","Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif;}#defence-architecture .solo rect{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo polygon{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo ellipse{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo circle{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo path{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo tspan{fill:#312e81!important;}#defence-architecture .data rect{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data polygon{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data ellipse{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data circle{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data path{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data tspan{fill:#7c2d12!important;}#defence-architecture .external rect{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external polygon{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external ellipse{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external circle{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external path{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external tspan{fill:#1e3a8a!important;}</style><g><marker id="defence-architecture_flowchart-v2-pointEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-pointStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="4.5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-circleEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="11" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-circleStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="-1" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="12" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="-1" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M310.665,223L248.09,255.5L248.09,284" id="L_agent_gateway_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_gateway_0" data-points="W3sieCI6MzEwLjY2NTAzOTA2MjUsInkiOjIyM30seyJ4IjoyNDguMDg5ODQzNzUsInkiOjI1NS41fSx7IngiOjI0OC4wODk4NDM3NSwieSI6Mjg4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M172.882,385L122.484,417.5L122.484,446" id="L_gateway_tools_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_tools_0" data-points="W3sieCI6MTcyLjg4MTYzMDk3OTkzODI4LCJ5IjozODV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDE3LjV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDUwfV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M122.484,525L122.484,557.5L122.484,586" id="L_tools_records_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_tools_records_0" data-points="W3sieCI6MTIyLjQ4NDM3NSwieSI6NTI1fSx7IngiOjEyMi40ODQzNzUsInkiOjU1Ny41fSx7IngiOjEyMi40ODQzNzUsInkiOjU5MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M455.069,223L517.645,255.5L517.645,295" id="L_agent_egress_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_egress_0" data-points="W3sieCI6NDU1LjA2OTMzNTkzNzUsInkiOjIyM30seyJ4Ijo1MTcuNjQ0NTMxMjUsInkiOjI1NS41fSx7IngiOjUxNy42NDQ1MzEyNSwieSI6Mjk5fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M382.867,83L382.867,115.5L382.867,144" id="L_user_agent_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_user_agent_0" data-points="W3sieCI6MzgyLjg2NzE4NzUsInkiOjgzfSx7IngiOjM4Mi44NjcxODc1LCJ5IjoxMTUuNX0seyJ4IjozODIuODY3MTg3NSwieSI6MTQ4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M323.298,385L373.695,417.5L373.695,446" id="L_gateway_claude_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_claude_0" data-points="W3sieCI6MzIzLjI5ODA1NjUyMDA2MTcsInkiOjM4NX0seyJ4IjozNzMuNjk1MzEyNSwieSI6NDE3LjV9LHsieCI6MzczLjY5NTMxMjUsInkiOjQ1MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(248.08984375, 255.5)"><g class="label" data-id="L_agent_gateway_0" transform="translate(-82.4140625, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="164.828125" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCP</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 417.5)"><g class="label" data-id="L_gateway_tools_0" transform="translate(-92.0546875, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="184.109375" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> workload</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 557.5)"><g class="label" data-id="L_tools_records_0" transform="translate(-90.9609375, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="181.921875" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Reads</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> changes</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(517.64453125, 255.5)"><g class="label" data-id="L_agent_egress_0" transform="translate(-99.828125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="199.65625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Outside-access</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(382.8671875, 115.5)"><g class="label" data-id="L_user_agent_0" transform="translate(-75.953125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="151.90625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">A2A</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> via</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> gateway</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(373.6953125, 417.5)"><g class="label" data-id="L_gateway_claude_0" transform="translate(-52.03125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="104.0625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g></g><g class="nodes"><g class="node default external" id="flowchart-user-0" transform="translate(382.8671875, 45.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-120.6171875" y="-37.5" width="241.234375" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Operator</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Investigate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incidents</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-agent-1" transform="translate(382.8671875, 185.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-108.953125" y="-37.5" width="217.90625" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Agent</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> response</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-gateway-2" transform="translate(248.08984375, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-116.1640625" y="-48.5" width="232.328125" height="97"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -33.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">agentgateway</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">JWT</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tool</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> access</tspan></tspan><tspan class="text-outer-tspan" x="0" y="2.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Content</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> rate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> limits</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-tools-3" transform="translate(122.484375, 487.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-114.484375" y="-37.5" width="228.96875" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCPServer</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tools</tspan></tspan></text></g></g></g><g class="node default data" id="flowchart-records-4" transform="translate(122.484375, 643.6671180725098)"><path d="M0,15.781710914454276 a107,15.781710914454276 0,0,0 214,0 a107,15.781710914454276 0,0,0 -214,0 l0,75.78171091445428 a107,15.781710914454276 0,0,0 214,0 l0,-75.78171091445428" class="basic label-container" style="fill:#fff7ed !important;stroke:#fb923c !important" transform="translate(-107, -53.67256637168141)"></path><g class="label" style="color:#7c2d12 !important" transform="translate(0, -14.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#7c2d12 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Three</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> records</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Status</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> history</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-egress-5" transform="translate(517.64453125, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-123.390625" y="-37.5" width="246.78125" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> egress</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> waypoint</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Deny</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> example.com</tspan></tspan></text></g></g></g><g class="node default external" id="flowchart-claude-17" transform="translate(373.6953125, 487.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-106.7265625" y="-37.5" width="213.453125" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Claude</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Haiku</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> 4.5</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Anthropic</tspan></tspan></text></g></g></g></g></g></g></svg>

### Without the control

Make an anonymous model request from the kagent runtime. This is a protocol check, separate from the agent's normal configured calls.

**Expected:** HTTP 200 without a token.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" -H 'Content-Type: application/json' -d '{"model":"claude-haiku-4-5","max_tokens":16,"messages":[{"role":"user","content":"Reply only READY"}]}'

### Add the control

Read and apply the public JWT policy. Require the lab issuer and audience on the gateway.

**Expected:** caller-identity is Accepted and Attached.

In [ ]:
cat "$DD_STATE/07-caller-identity.json"
kubectl --context kind-mesh1 apply -f "$DD_STATE/07-caller-identity.json"
dd_wait_policy caller-identity

### With the control

Repeat the anonymous request, then ask the kagent agent to investigate using its signed identity. Read the gateway's 401 log.

**Expected:** The anonymous request gets 401. The agent's authenticated model and tool calls still complete.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" -H 'Content-Type: application/json' -d '{"model":"claude-haiku-4-5","max_tokens":16,"messages":[{"role":"user","content":"Reply only READY"}]}'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
kubectl --context kind-mesh1 -n dd-gateway logs deploy/dd-gateway --since=30s | grep 'http.status=401'

## 5. Stop an agent closing unresolved incidents

Ask the agent to close every incident. Without a tool policy, all three records change to closed even though their faults remain unresolved. Restore the sample records, restrict Alice to investigation tools, then repeat the request.

A prompt is not the permission boundary. The gateway filters tools/list and refuses tools/call independently of the agent's behaviour.

<svg font-family="Arial, sans-serif" id="defence-architecture" width="100%" xmlns="http://www.w3.org/2000/svg" class="flowchart" style="max-width: 649.03515625px;" viewBox="0 0 649.03515625 705.334228515625" role="graphics-document document" aria-roledescription="flowchart-v2"><style>#defence-architecture{font-size:20px;fill:#1e293b;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#defence-architecture .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#defence-architecture .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#defence-architecture .error-icon{fill:#ffffff;}#defence-architecture .error-text{fill:#000000;stroke:#000000;}#defence-architecture .edge-thickness-normal{stroke-width:1px;}#defence-architecture .edge-thickness-thick{stroke-width:3.5px;}#defence-architecture .edge-pattern-solid{stroke-dasharray:0;}#defence-architecture .edge-thickness-invisible{stroke-width:0;fill:none;}#defence-architecture .edge-pattern-dashed{stroke-dasharray:3;}#defence-architecture .edge-pattern-dotted{stroke-dasharray:2;}#defence-architecture .marker{fill:#64748b;stroke:#64748b;}#defence-architecture .marker.cross{stroke:#64748b;}#defence-architecture svg{font-size:20px;}#defence-architecture p{margin:0;}#defence-architecture .label{color:#1e293b;}#defence-architecture .cluster-label text{fill:#000000;}#defence-architecture .cluster-label span{color:#000000;}#defence-architecture .cluster-label span p{background-color:transparent;}#defence-architecture .label text,#defence-architecture span{fill:#1e293b;color:#1e293b;}#defence-architecture .node rect,#defence-architecture .node circle,#defence-architecture .node ellipse,#defence-architecture .node polygon,#defence-architecture .node path{fill:#f5f3ff;stroke:#8b5cf6;stroke-width:1px;}#defence-architecture .rough-node .label text,#defence-architecture .node .label text,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-anchor:middle;}#defence-architecture .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#defence-architecture .rough-node .label,#defence-architecture .node .label,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-align:center;}#defence-architecture .node.clickable{cursor:pointer;}#defence-architecture .root .anchor path{fill:#64748b!important;stroke-width:0;stroke:#64748b;}#defence-architecture .arrowheadPath{fill:#0b0b0b;}#defence-architecture .edgePath .path{stroke:#64748b;stroke-width:2.0px;}#defence-architecture .flowchart-link{stroke:#64748b;fill:none;}#defence-architecture .edgeLabel{background-color:#eff6ff;text-align:center;}#defence-architecture .edgeLabel p{background-color:#eff6ff;}#defence-architecture .edgeLabel rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .labelBkg{background-color:rgba(239, 246, 255, 0.5);}#defence-architecture .cluster rect{fill:#f8fafc;stroke:#cbd5e1;stroke-width:1px;}#defence-architecture .cluster text{fill:#000000;}#defence-architecture .cluster span{color:#000000;}#defence-architecture div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-size:12px;background:#ffffff;border:1px solid hsl(0, 0%, 90%);border-radius:2px;pointer-events:none;z-index:100;}#defence-architecture .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#1e293b;}#defence-architecture rect.text{fill:none;stroke-width:0;}#defence-architecture .icon-shape,#defence-architecture .image-shape{background-color:#eff6ff;text-align:center;}#defence-architecture .icon-shape p,#defence-architecture .image-shape p{background-color:#eff6ff;padding:2px;}#defence-architecture .icon-shape rect,#defence-architecture .image-shape rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#defence-architecture .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#defence-architecture :root{--mermaid-font-family:Helvetica,"Liberation Sans",Arial,"Hiragino Kaku Gothic ProN","Noto Sans CJK JP","Microsoft YaHei","Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif;}#defence-architecture .solo rect{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo polygon{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo ellipse{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo circle{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo path{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo tspan{fill:#312e81!important;}#defence-architecture .data rect{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data polygon{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data ellipse{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data circle{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data path{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data tspan{fill:#7c2d12!important;}#defence-architecture .external rect{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external polygon{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external ellipse{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external circle{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external path{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external tspan{fill:#1e3a8a!important;}</style><g><marker id="defence-architecture_flowchart-v2-pointEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-pointStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="4.5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-circleEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="11" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-circleStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="-1" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="12" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="-1" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M310.665,223L248.09,255.5L248.09,284" id="L_agent_gateway_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_gateway_0" data-points="W3sieCI6MzEwLjY2NTAzOTA2MjUsInkiOjIyM30seyJ4IjoyNDguMDg5ODQzNzUsInkiOjI1NS41fSx7IngiOjI0OC4wODk4NDM3NSwieSI6Mjg4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M172.882,385L122.484,417.5L122.484,446" id="L_gateway_tools_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_tools_0" data-points="W3sieCI6MTcyLjg4MTYzMDk3OTkzODI4LCJ5IjozODV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDE3LjV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDUwfV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M122.484,525L122.484,557.5L122.484,586" id="L_tools_records_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_tools_records_0" data-points="W3sieCI6MTIyLjQ4NDM3NSwieSI6NTI1fSx7IngiOjEyMi40ODQzNzUsInkiOjU1Ny41fSx7IngiOjEyMi40ODQzNzUsInkiOjU5MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M455.069,223L517.645,255.5L517.645,295" id="L_agent_egress_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_egress_0" data-points="W3sieCI6NDU1LjA2OTMzNTkzNzUsInkiOjIyM30seyJ4Ijo1MTcuNjQ0NTMxMjUsInkiOjI1NS41fSx7IngiOjUxNy42NDQ1MzEyNSwieSI6Mjk5fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M382.867,83L382.867,115.5L382.867,144" id="L_user_agent_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_user_agent_0" data-points="W3sieCI6MzgyLjg2NzE4NzUsInkiOjgzfSx7IngiOjM4Mi44NjcxODc1LCJ5IjoxMTUuNX0seyJ4IjozODIuODY3MTg3NSwieSI6MTQ4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M323.298,385L373.695,417.5L373.695,446" id="L_gateway_claude_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_claude_0" data-points="W3sieCI6MzIzLjI5ODA1NjUyMDA2MTcsInkiOjM4NX0seyJ4IjozNzMuNjk1MzEyNSwieSI6NDE3LjV9LHsieCI6MzczLjY5NTMxMjUsInkiOjQ1MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(248.08984375, 255.5)"><g class="label" data-id="L_agent_gateway_0" transform="translate(-82.4140625, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="164.828125" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCP</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 417.5)"><g class="label" data-id="L_gateway_tools_0" transform="translate(-92.0546875, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="184.109375" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> workload</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 557.5)"><g class="label" data-id="L_tools_records_0" transform="translate(-90.9609375, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="181.921875" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Reads</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> changes</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(517.64453125, 255.5)"><g class="label" data-id="L_agent_egress_0" transform="translate(-99.828125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="199.65625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Outside-access</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(382.8671875, 115.5)"><g class="label" data-id="L_user_agent_0" transform="translate(-75.953125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="151.90625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">A2A</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> via</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> gateway</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(373.6953125, 417.5)"><g class="label" data-id="L_gateway_claude_0" transform="translate(-52.03125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="104.0625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g></g><g class="nodes"><g class="node default external" id="flowchart-user-0" transform="translate(382.8671875, 45.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-120.6171875" y="-37.5" width="241.234375" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Operator</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Investigate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incidents</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-agent-1" transform="translate(382.8671875, 185.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-108.953125" y="-37.5" width="217.90625" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Agent</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> response</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-gateway-2" transform="translate(248.08984375, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-116.1640625" y="-48.5" width="232.328125" height="97"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -33.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">agentgateway</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">JWT</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tool</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> access</tspan></tspan><tspan class="text-outer-tspan" x="0" y="2.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Content</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> rate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> limits</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-tools-3" transform="translate(122.484375, 487.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-114.484375" y="-37.5" width="228.96875" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCPServer</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tools</tspan></tspan></text></g></g></g><g class="node default data" id="flowchart-records-4" transform="translate(122.484375, 643.6671180725098)"><path d="M0,15.781710914454276 a107,15.781710914454276 0,0,0 214,0 a107,15.781710914454276 0,0,0 -214,0 l0,75.78171091445428 a107,15.781710914454276 0,0,0 214,0 l0,-75.78171091445428" class="basic label-container" style="fill:#fff7ed !important;stroke:#fb923c !important" transform="translate(-107, -53.67256637168141)"></path><g class="label" style="color:#7c2d12 !important" transform="translate(0, -14.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#7c2d12 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Three</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> records</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Status</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> history</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-egress-5" transform="translate(517.64453125, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-123.390625" y="-37.5" width="246.78125" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> egress</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> waypoint</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Deny</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> example.com</tspan></tspan></text></g></g></g><g class="node default external" id="flowchart-claude-17" transform="translate(373.6953125, 487.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-106.7265625" y="-37.5" width="213.453125" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Claude</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Haiku</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> 4.5</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Anthropic</tspan></tspan></text></g></g></g></g></g></g></svg>

### Without the control

Ask the agent to close every open incident. Watch the App tab: the open count falls to zero, but the unresolved count stays at three.

**Expected:** The real tool closes all three records. No underlying fault has been resolved.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use close_all_incidents to close every open incident now. Report the actual changed count.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

### Add the control

Permit only list_incidents and get_incident for Alice. As the administrator, restore the disposable records so the after check starts from the same state.

**Expected:** The policy is attached and the administrator restores three open incidents.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: tool-permissions, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: tools}
  backend:
    mcp:
      authorization:
        action: Allow
        policy:
          matchExpressions:
          - 'jwt.group == "admin" || mcp.tool.name in ["list_incidents", "get_incident"]'
YAML
dd_wait_policy tool-permissions
dd_mcp "$ADMIN" tools/call '{"name":"reset_demo_records","arguments":{}}'

### With the control

Repeat the same request to the agent. Check that write tools have disappeared, explicitly attempt bulk close, then ask the agent to read the records.

**Expected:** Bulk close is unavailable to the agent and the gateway refuses an explicit call. Investigation still works, and all three incidents remain open.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use close_all_incidents to close every open incident now. Report the actual changed count.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
printf 'Reader tools\n'; dd_mcp "$ALICE" tools/list
printf 'Reader call\n'; dd_mcp "$ALICE" tools/call '{"name":"close_all_incidents","arguments":{}}'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
kubectl --context kind-mesh1 -n dd-gateway logs deploy/dd-gateway --since=30s | grep 'Unknown tool: close_all_incidents'

## 6. Refuse personal data before the model call

Ask the agent to repeat a specimen email address. Add a request guard that rejects email addresses, then send the same message again.

This uses a real model. The proof is a failed agent turn plus the gateway's 403 and Reject counter, not an echo simulator claiming what the model received.

<svg font-family="Arial, sans-serif" id="defence-architecture" width="100%" xmlns="http://www.w3.org/2000/svg" class="flowchart" style="max-width: 649.03515625px;" viewBox="0 0 649.03515625 705.334228515625" role="graphics-document document" aria-roledescription="flowchart-v2"><style>#defence-architecture{font-size:20px;fill:#1e293b;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#defence-architecture .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#defence-architecture .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#defence-architecture .error-icon{fill:#ffffff;}#defence-architecture .error-text{fill:#000000;stroke:#000000;}#defence-architecture .edge-thickness-normal{stroke-width:1px;}#defence-architecture .edge-thickness-thick{stroke-width:3.5px;}#defence-architecture .edge-pattern-solid{stroke-dasharray:0;}#defence-architecture .edge-thickness-invisible{stroke-width:0;fill:none;}#defence-architecture .edge-pattern-dashed{stroke-dasharray:3;}#defence-architecture .edge-pattern-dotted{stroke-dasharray:2;}#defence-architecture .marker{fill:#64748b;stroke:#64748b;}#defence-architecture .marker.cross{stroke:#64748b;}#defence-architecture svg{font-size:20px;}#defence-architecture p{margin:0;}#defence-architecture .label{color:#1e293b;}#defence-architecture .cluster-label text{fill:#000000;}#defence-architecture .cluster-label span{color:#000000;}#defence-architecture .cluster-label span p{background-color:transparent;}#defence-architecture .label text,#defence-architecture span{fill:#1e293b;color:#1e293b;}#defence-architecture .node rect,#defence-architecture .node circle,#defence-architecture .node ellipse,#defence-architecture .node polygon,#defence-architecture .node path{fill:#f5f3ff;stroke:#8b5cf6;stroke-width:1px;}#defence-architecture .rough-node .label text,#defence-architecture .node .label text,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-anchor:middle;}#defence-architecture .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#defence-architecture .rough-node .label,#defence-architecture .node .label,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-align:center;}#defence-architecture .node.clickable{cursor:pointer;}#defence-architecture .root .anchor path{fill:#64748b!important;stroke-width:0;stroke:#64748b;}#defence-architecture .arrowheadPath{fill:#0b0b0b;}#defence-architecture .edgePath .path{stroke:#64748b;stroke-width:2.0px;}#defence-architecture .flowchart-link{stroke:#64748b;fill:none;}#defence-architecture .edgeLabel{background-color:#eff6ff;text-align:center;}#defence-architecture .edgeLabel p{background-color:#eff6ff;}#defence-architecture .edgeLabel rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .labelBkg{background-color:rgba(239, 246, 255, 0.5);}#defence-architecture .cluster rect{fill:#f8fafc;stroke:#cbd5e1;stroke-width:1px;}#defence-architecture .cluster text{fill:#000000;}#defence-architecture .cluster span{color:#000000;}#defence-architecture div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-size:12px;background:#ffffff;border:1px solid hsl(0, 0%, 90%);border-radius:2px;pointer-events:none;z-index:100;}#defence-architecture .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#1e293b;}#defence-architecture rect.text{fill:none;stroke-width:0;}#defence-architecture .icon-shape,#defence-architecture .image-shape{background-color:#eff6ff;text-align:center;}#defence-architecture .icon-shape p,#defence-architecture .image-shape p{background-color:#eff6ff;padding:2px;}#defence-architecture .icon-shape rect,#defence-architecture .image-shape rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#defence-architecture .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#defence-architecture :root{--mermaid-font-family:Helvetica,"Liberation Sans",Arial,"Hiragino Kaku Gothic ProN","Noto Sans CJK JP","Microsoft YaHei","Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif;}#defence-architecture .solo rect{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo polygon{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo ellipse{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo circle{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo path{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo tspan{fill:#312e81!important;}#defence-architecture .data rect{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data polygon{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data ellipse{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data circle{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data path{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data tspan{fill:#7c2d12!important;}#defence-architecture .external rect{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external polygon{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external ellipse{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external circle{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external path{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external tspan{fill:#1e3a8a!important;}</style><g><marker id="defence-architecture_flowchart-v2-pointEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-pointStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="4.5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-circleEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="11" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-circleStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="-1" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="12" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="-1" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M310.665,223L248.09,255.5L248.09,284" id="L_agent_gateway_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_gateway_0" data-points="W3sieCI6MzEwLjY2NTAzOTA2MjUsInkiOjIyM30seyJ4IjoyNDguMDg5ODQzNzUsInkiOjI1NS41fSx7IngiOjI0OC4wODk4NDM3NSwieSI6Mjg4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M172.882,385L122.484,417.5L122.484,446" id="L_gateway_tools_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_tools_0" data-points="W3sieCI6MTcyLjg4MTYzMDk3OTkzODI4LCJ5IjozODV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDE3LjV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDUwfV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M122.484,525L122.484,557.5L122.484,586" id="L_tools_records_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_tools_records_0" data-points="W3sieCI6MTIyLjQ4NDM3NSwieSI6NTI1fSx7IngiOjEyMi40ODQzNzUsInkiOjU1Ny41fSx7IngiOjEyMi40ODQzNzUsInkiOjU5MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M455.069,223L517.645,255.5L517.645,295" id="L_agent_egress_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_egress_0" data-points="W3sieCI6NDU1LjA2OTMzNTkzNzUsInkiOjIyM30seyJ4Ijo1MTcuNjQ0NTMxMjUsInkiOjI1NS41fSx7IngiOjUxNy42NDQ1MzEyNSwieSI6Mjk5fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M382.867,83L382.867,115.5L382.867,144" id="L_user_agent_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_user_agent_0" data-points="W3sieCI6MzgyLjg2NzE4NzUsInkiOjgzfSx7IngiOjM4Mi44NjcxODc1LCJ5IjoxMTUuNX0seyJ4IjozODIuODY3MTg3NSwieSI6MTQ4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M323.298,385L373.695,417.5L373.695,446" id="L_gateway_claude_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_claude_0" data-points="W3sieCI6MzIzLjI5ODA1NjUyMDA2MTcsInkiOjM4NX0seyJ4IjozNzMuNjk1MzEyNSwieSI6NDE3LjV9LHsieCI6MzczLjY5NTMxMjUsInkiOjQ1MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(248.08984375, 255.5)"><g class="label" data-id="L_agent_gateway_0" transform="translate(-82.4140625, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="164.828125" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCP</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 417.5)"><g class="label" data-id="L_gateway_tools_0" transform="translate(-92.0546875, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="184.109375" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> workload</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 557.5)"><g class="label" data-id="L_tools_records_0" transform="translate(-90.9609375, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="181.921875" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Reads</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> changes</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(517.64453125, 255.5)"><g class="label" data-id="L_agent_egress_0" transform="translate(-99.828125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="199.65625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Outside-access</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(382.8671875, 115.5)"><g class="label" data-id="L_user_agent_0" transform="translate(-75.953125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="151.90625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">A2A</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> via</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> gateway</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(373.6953125, 417.5)"><g class="label" data-id="L_gateway_claude_0" transform="translate(-52.03125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="104.0625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g></g><g class="nodes"><g class="node default external" id="flowchart-user-0" transform="translate(382.8671875, 45.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-120.6171875" y="-37.5" width="241.234375" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Operator</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Investigate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incidents</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-agent-1" transform="translate(382.8671875, 185.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-108.953125" y="-37.5" width="217.90625" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Agent</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> response</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-gateway-2" transform="translate(248.08984375, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-116.1640625" y="-48.5" width="232.328125" height="97"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -33.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">agentgateway</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">JWT</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tool</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> access</tspan></tspan><tspan class="text-outer-tspan" x="0" y="2.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Content</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> rate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> limits</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-tools-3" transform="translate(122.484375, 487.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-114.484375" y="-37.5" width="228.96875" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCPServer</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tools</tspan></tspan></text></g></g></g><g class="node default data" id="flowchart-records-4" transform="translate(122.484375, 643.6671180725098)"><path d="M0,15.781710914454276 a107,15.781710914454276 0,0,0 214,0 a107,15.781710914454276 0,0,0 -214,0 l0,75.78171091445428 a107,15.781710914454276 0,0,0 214,0 l0,-75.78171091445428" class="basic label-container" style="fill:#fff7ed !important;stroke:#fb923c !important" transform="translate(-107, -53.67256637168141)"></path><g class="label" style="color:#7c2d12 !important" transform="translate(0, -14.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#7c2d12 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Three</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> records</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Status</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> history</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-egress-5" transform="translate(517.64453125, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-123.390625" y="-37.5" width="246.78125" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> egress</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> waypoint</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Deny</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> example.com</tspan></tspan></text></g></g></g><g class="node default external" id="flowchart-claude-17" transform="translate(373.6953125, 487.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-106.7265625" y="-37.5" width="213.453125" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Claude</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Haiku</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> 4.5</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Anthropic</tspan></tspan></text></g></g></g></g></g></g></svg>

### Without the control

Ask the agent to repeat the specimen email.

**Expected:** The agent completes and includes alice@example.test in its reply.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Repeat this sample email exactly: alice@example.test' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

### Add the control

Reject recognised email addresses on the model route before the provider call.

**Expected:** reject-personal-data is Accepted and Attached.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: reject-personal-data, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, name: model}
  backend:
    ai:
      promptGuard:
        request:
        - regex:
            action: Reject
            builtins: [Email]
          response:
            statusCode: 403
            message: "Personal data refused before the model call"
YAML
dd_wait_policy reject-personal-data

### With the control

Send the same message through the agent. Read the gateway's guardrail evidence, then send a message without personal data.

**Expected:** The email turn fails with a gateway 403 and a Reject counter. The clean turn completes.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Repeat this sample email exactly: alice@example.test' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Reply only READY' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

## 7. Limit model requests per caller

Send a bounded sequence of model requests from the kagent runtime. Add a global limit of three requests per minute per signed subject, then repeat the check.

This is a protocol-level rate check. Agent turns can make multiple model requests, so a request allowance is not a turn allowance. Replies are capped at sixteen tokens and use the real provider.

<svg font-family="Arial, sans-serif" id="defence-architecture" width="100%" xmlns="http://www.w3.org/2000/svg" class="flowchart" style="max-width: 649.03515625px;" viewBox="0 0 649.03515625 705.334228515625" role="graphics-document document" aria-roledescription="flowchart-v2"><style>#defence-architecture{font-size:20px;fill:#1e293b;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#defence-architecture .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#defence-architecture .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#defence-architecture .error-icon{fill:#ffffff;}#defence-architecture .error-text{fill:#000000;stroke:#000000;}#defence-architecture .edge-thickness-normal{stroke-width:1px;}#defence-architecture .edge-thickness-thick{stroke-width:3.5px;}#defence-architecture .edge-pattern-solid{stroke-dasharray:0;}#defence-architecture .edge-thickness-invisible{stroke-width:0;fill:none;}#defence-architecture .edge-pattern-dashed{stroke-dasharray:3;}#defence-architecture .edge-pattern-dotted{stroke-dasharray:2;}#defence-architecture .marker{fill:#64748b;stroke:#64748b;}#defence-architecture .marker.cross{stroke:#64748b;}#defence-architecture svg{font-size:20px;}#defence-architecture p{margin:0;}#defence-architecture .label{color:#1e293b;}#defence-architecture .cluster-label text{fill:#000000;}#defence-architecture .cluster-label span{color:#000000;}#defence-architecture .cluster-label span p{background-color:transparent;}#defence-architecture .label text,#defence-architecture span{fill:#1e293b;color:#1e293b;}#defence-architecture .node rect,#defence-architecture .node circle,#defence-architecture .node ellipse,#defence-architecture .node polygon,#defence-architecture .node path{fill:#f5f3ff;stroke:#8b5cf6;stroke-width:1px;}#defence-architecture .rough-node .label text,#defence-architecture .node .label text,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-anchor:middle;}#defence-architecture .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#defence-architecture .rough-node .label,#defence-architecture .node .label,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-align:center;}#defence-architecture .node.clickable{cursor:pointer;}#defence-architecture .root .anchor path{fill:#64748b!important;stroke-width:0;stroke:#64748b;}#defence-architecture .arrowheadPath{fill:#0b0b0b;}#defence-architecture .edgePath .path{stroke:#64748b;stroke-width:2.0px;}#defence-architecture .flowchart-link{stroke:#64748b;fill:none;}#defence-architecture .edgeLabel{background-color:#eff6ff;text-align:center;}#defence-architecture .edgeLabel p{background-color:#eff6ff;}#defence-architecture .edgeLabel rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .labelBkg{background-color:rgba(239, 246, 255, 0.5);}#defence-architecture .cluster rect{fill:#f8fafc;stroke:#cbd5e1;stroke-width:1px;}#defence-architecture .cluster text{fill:#000000;}#defence-architecture .cluster span{color:#000000;}#defence-architecture div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-size:12px;background:#ffffff;border:1px solid hsl(0, 0%, 90%);border-radius:2px;pointer-events:none;z-index:100;}#defence-architecture .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#1e293b;}#defence-architecture rect.text{fill:none;stroke-width:0;}#defence-architecture .icon-shape,#defence-architecture .image-shape{background-color:#eff6ff;text-align:center;}#defence-architecture .icon-shape p,#defence-architecture .image-shape p{background-color:#eff6ff;padding:2px;}#defence-architecture .icon-shape rect,#defence-architecture .image-shape rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#defence-architecture .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#defence-architecture :root{--mermaid-font-family:Helvetica,"Liberation Sans",Arial,"Hiragino Kaku Gothic ProN","Noto Sans CJK JP","Microsoft YaHei","Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif;}#defence-architecture .solo rect{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo polygon{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo ellipse{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo circle{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo path{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo tspan{fill:#312e81!important;}#defence-architecture .data rect{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data polygon{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data ellipse{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data circle{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data path{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data tspan{fill:#7c2d12!important;}#defence-architecture .external rect{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external polygon{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external ellipse{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external circle{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external path{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external tspan{fill:#1e3a8a!important;}</style><g><marker id="defence-architecture_flowchart-v2-pointEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-pointStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="4.5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-circleEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="11" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-circleStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="-1" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="12" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="-1" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M310.665,223L248.09,255.5L248.09,284" id="L_agent_gateway_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_gateway_0" data-points="W3sieCI6MzEwLjY2NTAzOTA2MjUsInkiOjIyM30seyJ4IjoyNDguMDg5ODQzNzUsInkiOjI1NS41fSx7IngiOjI0OC4wODk4NDM3NSwieSI6Mjg4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M172.882,385L122.484,417.5L122.484,446" id="L_gateway_tools_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_tools_0" data-points="W3sieCI6MTcyLjg4MTYzMDk3OTkzODI4LCJ5IjozODV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDE3LjV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDUwfV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M122.484,525L122.484,557.5L122.484,586" id="L_tools_records_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_tools_records_0" data-points="W3sieCI6MTIyLjQ4NDM3NSwieSI6NTI1fSx7IngiOjEyMi40ODQzNzUsInkiOjU1Ny41fSx7IngiOjEyMi40ODQzNzUsInkiOjU5MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M455.069,223L517.645,255.5L517.645,295" id="L_agent_egress_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_egress_0" data-points="W3sieCI6NDU1LjA2OTMzNTkzNzUsInkiOjIyM30seyJ4Ijo1MTcuNjQ0NTMxMjUsInkiOjI1NS41fSx7IngiOjUxNy42NDQ1MzEyNSwieSI6Mjk5fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M382.867,83L382.867,115.5L382.867,144" id="L_user_agent_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_user_agent_0" data-points="W3sieCI6MzgyLjg2NzE4NzUsInkiOjgzfSx7IngiOjM4Mi44NjcxODc1LCJ5IjoxMTUuNX0seyJ4IjozODIuODY3MTg3NSwieSI6MTQ4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M323.298,385L373.695,417.5L373.695,446" id="L_gateway_claude_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_claude_0" data-points="W3sieCI6MzIzLjI5ODA1NjUyMDA2MTcsInkiOjM4NX0seyJ4IjozNzMuNjk1MzEyNSwieSI6NDE3LjV9LHsieCI6MzczLjY5NTMxMjUsInkiOjQ1MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(248.08984375, 255.5)"><g class="label" data-id="L_agent_gateway_0" transform="translate(-82.4140625, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="164.828125" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCP</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 417.5)"><g class="label" data-id="L_gateway_tools_0" transform="translate(-92.0546875, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="184.109375" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> workload</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 557.5)"><g class="label" data-id="L_tools_records_0" transform="translate(-90.9609375, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="181.921875" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Reads</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> changes</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(517.64453125, 255.5)"><g class="label" data-id="L_agent_egress_0" transform="translate(-99.828125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="199.65625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Outside-access</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(382.8671875, 115.5)"><g class="label" data-id="L_user_agent_0" transform="translate(-75.953125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="151.90625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">A2A</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> via</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> gateway</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(373.6953125, 417.5)"><g class="label" data-id="L_gateway_claude_0" transform="translate(-52.03125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="104.0625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g></g><g class="nodes"><g class="node default external" id="flowchart-user-0" transform="translate(382.8671875, 45.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-120.6171875" y="-37.5" width="241.234375" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Operator</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Investigate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incidents</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-agent-1" transform="translate(382.8671875, 185.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-108.953125" y="-37.5" width="217.90625" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Agent</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> response</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-gateway-2" transform="translate(248.08984375, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-116.1640625" y="-48.5" width="232.328125" height="97"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -33.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">agentgateway</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">JWT</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tool</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> access</tspan></tspan><tspan class="text-outer-tspan" x="0" y="2.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Content</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> rate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> limits</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-tools-3" transform="translate(122.484375, 487.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-114.484375" y="-37.5" width="228.96875" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCPServer</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tools</tspan></tspan></text></g></g></g><g class="node default data" id="flowchart-records-4" transform="translate(122.484375, 643.6671180725098)"><path d="M0,15.781710914454276 a107,15.781710914454276 0,0,0 214,0 a107,15.781710914454276 0,0,0 -214,0 l0,75.78171091445428 a107,15.781710914454276 0,0,0 214,0 l0,-75.78171091445428" class="basic label-container" style="fill:#fff7ed !important;stroke:#fb923c !important" transform="translate(-107, -53.67256637168141)"></path><g class="label" style="color:#7c2d12 !important" transform="translate(0, -14.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#7c2d12 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Three</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> records</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Status</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> history</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-egress-5" transform="translate(517.64453125, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-123.390625" y="-37.5" width="246.78125" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> egress</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> waypoint</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Deny</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> example.com</tspan></tspan></text></g></g></g><g class="node default external" id="flowchart-claude-17" transform="translate(373.6953125, 487.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-106.7265625" y="-37.5" width="213.453125" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Claude</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Haiku</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> 4.5</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Anthropic</tspan></tspan></text></g></g></g></g></g></g></svg>

### Without the control

Send the bounded rate check with fresh Alice and Bob subjects.

**Expected:** All requests return 200.

In [ ]:
dd_burst open "${N:-6}"

### Add the control

Apply the per-subject descriptor and wait for the limiter to load the current generation.

**Expected:** The policy is attached and the limiter reports ACCEPTED.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: ratelimit.solo.io/v1alpha1
kind: RateLimitConfig
metadata: {name: dd-per-person, namespace: dd-gateway}
spec:
  raw:
    descriptors:
    - key: dd-sub
      rateLimit: {requestsPerUnit: 3, unit: MINUTE}
    rateLimits:
    - actions:
      - cel: {expression: jwt.sub, key: dd-sub}
      type: REQUEST
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: per-person-rate, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, name: model}
  traffic:
    entRateLimit:
      global:
        rateLimitConfigRefs: [{name: dd-per-person}]
        backendRef:
          name: dd-rate-limiter
          namespace: agentgateway-system
          port: 8083
YAML
dd_wait_policy per-person-rate
dd_wait_rate

### With the control

Repeat the bounded check and read the 429 metric. A request as Bob checks that the allowance is per identity.

**Expected:** Alice reaches 429 while Bob still gets 200.

In [ ]:
dd_burst limited "${N:-6}"
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

## 8. Read the recorded outcomes by caller

The platform already records refusals. Add caller identity to the gateway metrics and generate another bounded rate check.

The live board shows the kagent Agent and its last A2A task alongside policy state, gateway counters and workload refusal evidence. Detection observes; it does not add another refusal control.

<svg font-family="Arial, sans-serif" id="defence-architecture" width="100%" xmlns="http://www.w3.org/2000/svg" class="flowchart" style="max-width: 649.03515625px;" viewBox="0 0 649.03515625 705.334228515625" role="graphics-document document" aria-roledescription="flowchart-v2"><style>#defence-architecture{font-size:20px;fill:#1e293b;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#defence-architecture .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#defence-architecture .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#defence-architecture .error-icon{fill:#ffffff;}#defence-architecture .error-text{fill:#000000;stroke:#000000;}#defence-architecture .edge-thickness-normal{stroke-width:1px;}#defence-architecture .edge-thickness-thick{stroke-width:3.5px;}#defence-architecture .edge-pattern-solid{stroke-dasharray:0;}#defence-architecture .edge-thickness-invisible{stroke-width:0;fill:none;}#defence-architecture .edge-pattern-dashed{stroke-dasharray:3;}#defence-architecture .edge-pattern-dotted{stroke-dasharray:2;}#defence-architecture .marker{fill:#64748b;stroke:#64748b;}#defence-architecture .marker.cross{stroke:#64748b;}#defence-architecture svg{font-size:20px;}#defence-architecture p{margin:0;}#defence-architecture .label{color:#1e293b;}#defence-architecture .cluster-label text{fill:#000000;}#defence-architecture .cluster-label span{color:#000000;}#defence-architecture .cluster-label span p{background-color:transparent;}#defence-architecture .label text,#defence-architecture span{fill:#1e293b;color:#1e293b;}#defence-architecture .node rect,#defence-architecture .node circle,#defence-architecture .node ellipse,#defence-architecture .node polygon,#defence-architecture .node path{fill:#f5f3ff;stroke:#8b5cf6;stroke-width:1px;}#defence-architecture .rough-node .label text,#defence-architecture .node .label text,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-anchor:middle;}#defence-architecture .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#defence-architecture .rough-node .label,#defence-architecture .node .label,#defence-architecture .image-shape .label,#defence-architecture .icon-shape .label{text-align:center;}#defence-architecture .node.clickable{cursor:pointer;}#defence-architecture .root .anchor path{fill:#64748b!important;stroke-width:0;stroke:#64748b;}#defence-architecture .arrowheadPath{fill:#0b0b0b;}#defence-architecture .edgePath .path{stroke:#64748b;stroke-width:2.0px;}#defence-architecture .flowchart-link{stroke:#64748b;fill:none;}#defence-architecture .edgeLabel{background-color:#eff6ff;text-align:center;}#defence-architecture .edgeLabel p{background-color:#eff6ff;}#defence-architecture .edgeLabel rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .labelBkg{background-color:rgba(239, 246, 255, 0.5);}#defence-architecture .cluster rect{fill:#f8fafc;stroke:#cbd5e1;stroke-width:1px;}#defence-architecture .cluster text{fill:#000000;}#defence-architecture .cluster span{color:#000000;}#defence-architecture div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-size:12px;background:#ffffff;border:1px solid hsl(0, 0%, 90%);border-radius:2px;pointer-events:none;z-index:100;}#defence-architecture .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#1e293b;}#defence-architecture rect.text{fill:none;stroke-width:0;}#defence-architecture .icon-shape,#defence-architecture .image-shape{background-color:#eff6ff;text-align:center;}#defence-architecture .icon-shape p,#defence-architecture .image-shape p{background-color:#eff6ff;padding:2px;}#defence-architecture .icon-shape rect,#defence-architecture .image-shape rect{opacity:0.5;background-color:#eff6ff;fill:#eff6ff;}#defence-architecture .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#defence-architecture .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#defence-architecture :root{--mermaid-font-family:Helvetica,"Liberation Sans",Arial,"Hiragino Kaku Gothic ProN","Noto Sans CJK JP","Microsoft YaHei","Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif;}#defence-architecture .solo rect{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo polygon{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo ellipse{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo circle{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo path{fill:#f5f3ff!important;stroke:#8b5cf6!important;stroke-width:2px!important;color:#312e81!important;}#defence-architecture .solo tspan{fill:#312e81!important;}#defence-architecture .data rect{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data polygon{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data ellipse{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data circle{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data path{fill:#fff7ed!important;stroke:#fb923c!important;color:#7c2d12!important;}#defence-architecture .data tspan{fill:#7c2d12!important;}#defence-architecture .external rect{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external polygon{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external ellipse{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external circle{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external path{fill:#eff6ff!important;stroke:#60a5fa!important;color:#1e3a8a!important;}#defence-architecture .external tspan{fill:#1e3a8a!important;}</style><g><marker id="defence-architecture_flowchart-v2-pointEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-pointStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="4.5" refY="5" markerUnits="userSpaceOnUse" markerWidth="8" markerHeight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-circleEnd" class="marker flowchart-v2" viewBox="0 0 10 10" refX="11" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-circleStart" class="marker flowchart-v2" viewBox="0 0 10 10" refX="-1" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="defence-architecture_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="12" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="defence-architecture_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewBox="0 0 11 11" refX="-1" refY="5.2" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M310.665,223L248.09,255.5L248.09,284" id="L_agent_gateway_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_gateway_0" data-points="W3sieCI6MzEwLjY2NTAzOTA2MjUsInkiOjIyM30seyJ4IjoyNDguMDg5ODQzNzUsInkiOjI1NS41fSx7IngiOjI0OC4wODk4NDM3NSwieSI6Mjg4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M172.882,385L122.484,417.5L122.484,446" id="L_gateway_tools_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_tools_0" data-points="W3sieCI6MTcyLjg4MTYzMDk3OTkzODI4LCJ5IjozODV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDE3LjV9LHsieCI6MTIyLjQ4NDM3NSwieSI6NDUwfV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M122.484,525L122.484,557.5L122.484,586" id="L_tools_records_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_tools_records_0" data-points="W3sieCI6MTIyLjQ4NDM3NSwieSI6NTI1fSx7IngiOjEyMi40ODQzNzUsInkiOjU1Ny41fSx7IngiOjEyMi40ODQzNzUsInkiOjU5MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M455.069,223L517.645,255.5L517.645,295" id="L_agent_egress_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_agent_egress_0" data-points="W3sieCI6NDU1LjA2OTMzNTkzNzUsInkiOjIyM30seyJ4Ijo1MTcuNjQ0NTMxMjUsInkiOjI1NS41fSx7IngiOjUxNy42NDQ1MzEyNSwieSI6Mjk5fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M382.867,83L382.867,115.5L382.867,144" id="L_user_agent_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_user_agent_0" data-points="W3sieCI6MzgyLjg2NzE4NzUsInkiOjgzfSx7IngiOjM4Mi44NjcxODc1LCJ5IjoxMTUuNX0seyJ4IjozODIuODY3MTg3NSwieSI6MTQ4fV0=" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path><path d="M323.298,385L373.695,417.5L373.695,446" id="L_gateway_claude_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_gateway_claude_0" data-points="W3sieCI6MzIzLjI5ODA1NjUyMDA2MTcsInkiOjM4NX0seyJ4IjozNzMuNjk1MzEyNSwieSI6NDE3LjV9LHsieCI6MzczLjY5NTMxMjUsInkiOjQ1MH1d" marker-end="url(#defence-architecture_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(248.08984375, 255.5)"><g class="label" data-id="L_agent_gateway_0" transform="translate(-82.4140625, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="164.828125" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCP</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 417.5)"><g class="label" data-id="L_gateway_tools_0" transform="translate(-92.0546875, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="184.109375" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> workload</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(122.484375, 557.5)"><g class="label" data-id="L_tools_records_0" transform="translate(-90.9609375, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="181.921875" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Reads</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> changes</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(517.64453125, 255.5)"><g class="label" data-id="L_agent_egress_0" transform="translate(-99.828125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="199.65625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Outside-access</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> check</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(382.8671875, 115.5)"><g class="label" data-id="L_user_agent_0" transform="translate(-75.953125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="151.90625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">A2A</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> via</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> gateway</tspan></tspan></text></g></g></g><g class="edgeLabel" transform="translate(373.6953125, 417.5)"><g class="label" data-id="L_gateway_claude_0" transform="translate(-52.03125, -13.5)"><g><rect class="background" style="" x="-2" y="0" width="104.0625" height="27"></rect><text y="-10.1" style=""><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Model</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> calls</tspan></tspan></text></g></g></g></g><g class="nodes"><g class="node default external" id="flowchart-user-0" transform="translate(382.8671875, 45.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-120.6171875" y="-37.5" width="241.234375" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Operator</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Investigate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incidents</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-agent-1" transform="translate(382.8671875, 185.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-108.953125" y="-37.5" width="217.90625" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Agent</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> response</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-gateway-2" transform="translate(248.08984375, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-116.1640625" y="-48.5" width="232.328125" height="97"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -33.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">agentgateway</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">JWT</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tool</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> access</tspan></tspan><tspan class="text-outer-tspan" x="0" y="2.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Content</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> /</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> rate</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> limits</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-tools-3" transform="translate(122.484375, 487.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-114.484375" y="-37.5" width="228.96875" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">kagent</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> MCPServer</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> tools</tspan></tspan></text></g></g></g><g class="node default data" id="flowchart-records-4" transform="translate(122.484375, 643.6671180725098)"><path d="M0,15.781710914454276 a107,15.781710914454276 0,0,0 214,0 a107,15.781710914454276 0,0,0 -214,0 l0,75.78171091445428 a107,15.781710914454276 0,0,0 214,0 l0,-75.78171091445428" class="basic label-container" style="fill:#fff7ed !important;stroke:#fb923c !important" transform="translate(-107, -53.67256637168141)"></path><g class="label" style="color:#7c2d12 !important" transform="translate(0, -14.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#7c2d12 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Three</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> incident</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> records</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Status</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> and</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> history</tspan></tspan></text></g></g></g><g class="node default solo" id="flowchart-egress-5" transform="translate(517.64453125, 336.5)"><rect class="basic label-container" style="fill:#f5f3ff !important;stroke:#8b5cf6 !important;stroke-width:2px !important" x="-123.390625" y="-37.5" width="246.78125" height="75"></rect><g class="label" style="color:#312e81 !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#312e81 !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Istio</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> egress</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> waypoint</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Deny</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> example.com</tspan></tspan></text></g></g></g><g class="node default external" id="flowchart-claude-17" transform="translate(373.6953125, 487.5)"><rect class="basic label-container" style="fill:#eff6ff !important;stroke:#60a5fa !important" x="-106.7265625" y="-37.5" width="213.453125" height="75"></rect><g class="label" style="color:#1e3a8a !important" transform="translate(0, -22.5)"><rect></rect><g><rect class="background" style="stroke: none"></rect><text y="-10.1" style="fill:#1e3a8a !important"><tspan class="text-outer-tspan" x="0" y="-0.1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Claude</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> Haiku</tspan><tspan font-style="normal" class="text-inner-tspan" font-weight="normal"> 4.5</tspan></tspan><tspan class="text-outer-tspan" x="0" y="1em" dy="1.1em"><tspan font-style="normal" class="text-inner-tspan" font-weight="normal">Anthropic</tspan></tspan></text></g></g></g></g></g></g></svg>

### Before adding caller labels

Read the gateway's own metrics.

**Expected:** 401, 403 and 429 outcomes are already recorded.

In [ ]:
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

### Add caller labels

Attach a metric attribute based on the validated JWT subject.

**Expected:** identity-metrics is Accepted and Attached.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
# These are bounded demonstration identities, not a production label-cardinality plan.
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: identity-metrics, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: Gateway, name: dd-gateway}
  frontend:
    metrics:
      attributes:
        add:
        - {name: caller, expression: jwt.sub}
YAML
dd_wait_policy identity-metrics

### With caller labels

Generate new rate refusals and read the identity-labelled metrics.

**Expected:** The 429 samples name the Alice subject. Bob remains allowed.

In [ ]:
dd_burst limited 6
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

## Reset

Remove only this lab's kagent resources, gateways and credentials. The console checks that they are gone.

In [ ]:
kubectl --context kind-mesh1 delete namespace dd-agents dd-tools dd-gateway dd-models --ignore-not-found --wait=true --timeout=180s
kubectl --context kind-mesh1 -n agentgateway-system delete service dd-rate-limiter --ignore-not-found
rm -f "$DD_STATE/private.pem" "$DD_STATE/07-caller-identity.json" "$DD_STATE/last-agent-task.json"